# Data reconnaissance: Binance monthly archives

Closes #7.

Question: what does one month of Binance data cost in bytes, minutes and disk, and can we verify
we downloaded it intact?

This is a spike. It measures the cost and mechanics of pulling data from `data.binance.vision`,
not the phenomenon the project is trying to measure. The estimand is still undecided (#9) and the
release calendar is still undecided (#8); the sample counts below use a cadence-based estimate
where the calendar itself has not been fetched, and that is flagged where it applies.

In [1]:
import hashlib
import io
import re
import time
import zipfile
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd
import requests

SYMBOL = "BTCUSDT"
MONTH = "2024-06"
BASE = "https://data.binance.vision"
S3_LIST = "https://s3-ap-northeast-1.amazonaws.com/data.binance.vision/"

## Endpoints

Three monthly products, one file per symbol-month:

- klines: `{base}/data/spot/monthly/klines/{symbol}/{interval}/{symbol}-{interval}-{yyyy-mm}.zip`
- aggTrades: `{base}/data/spot/monthly/aggTrades/{symbol}/{symbol}-aggTrades-{yyyy-mm}.zip`
- trades: `{base}/data/spot/monthly/trades/{symbol}/{symbol}-trades-{yyyy-mm}.zip`

Each has a checksum sidecar at the same path with `.CHECKSUM` appended, one line:
`<sha256 hex>  <filename>`.

The same three products also exist under `data/spot/daily/...` with a `yyyy-mm-dd` file per day
instead of a month; `#6` already used the daily klines path for the kill-check and the health
check below reuses it.

One working example per product, verified live with a HEAD request:

In [2]:
PRODUCTS = {
    "klines": {
        "url": f"{BASE}/data/spot/monthly/klines/{SYMBOL}/1s/{SYMBOL}-1s-{MONTH}.zip",
        "columns": [
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_volume",
            "trades",
            "taker_buy_base",
            "taker_buy_quote",
            "ignore",
        ],
    },
    "aggTrades": {
        "url": f"{BASE}/data/spot/monthly/aggTrades/{SYMBOL}/{SYMBOL}-aggTrades-{MONTH}.zip",
        "columns": [
            "agg_trade_id",
            "price",
            "quantity",
            "first_trade_id",
            "last_trade_id",
            "transact_time",
            "is_buyer_maker",
            "is_best_match",
        ],
    },
    "trades": {
        "url": f"{BASE}/data/spot/monthly/trades/{SYMBOL}/{SYMBOL}-trades-{MONTH}.zip",
        "columns": [
            "trade_id",
            "price",
            "qty",
            "quote_qty",
            "time",
            "is_buyer_maker",
            "is_best_match",
        ],
    },
}

for name, cfg in PRODUCTS.items():
    head = requests.head(cfg["url"], timeout=30)
    print(f"{name:10s} {head.status_code}  {cfg['url']}")

klines     200  https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1s/BTCUSDT-1s-2024-06.zip


aggTrades  200  https://data.binance.vision/data/spot/monthly/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2024-06.zip


trades     200  https://data.binance.vision/data/spot/monthly/trades/BTCUSDT/BTCUSDT-trades-2024-06.zip


## Checksum verification

`verify_checksum` computes the sha256 of the downloaded bytes and compares it against the
`.CHECKSUM` sidecar. First, a real file against its real checksum:

In [3]:
def verify_checksum(data: bytes, expected_hex: str) -> None:
    actual_hex = hashlib.sha256(data).hexdigest()
    if actual_hex != expected_hex:
        raise ValueError(f"checksum mismatch: expected {expected_hex}, got {actual_hex}")


klines_resp = requests.get(PRODUCTS["klines"]["url"], timeout=120)
klines_resp.raise_for_status()
klines_bytes = klines_resp.content

checksum_resp = requests.get(PRODUCTS["klines"]["url"] + ".CHECKSUM", timeout=30)
checksum_resp.raise_for_status()
expected_hex = checksum_resp.text.split()[0]

verify_checksum(klines_bytes, expected_hex)
print("checksum matched:", expected_hex)

checksum matched: 232948f6e3086abc87338a81daef5f9375e3d4209678d826c39724a9c0f7ae05


Now the failure mode: flip one byte and confirm the mismatch is caught, not silently accepted.

In [4]:
corrupted = bytearray(klines_bytes)
corrupted[1_000_000] ^= 0xFF

try:
    verify_checksum(bytes(corrupted), expected_hex)
    print("did not catch corruption")
except ValueError as e:
    print("caught:", e)

caught: checksum mismatch: expected 232948f6e3086abc87338a81daef5f9375e3d4209678d826c39724a9c0f7ae05, got ce6442212c5ccdd4b8e50ccc6ad82febc9e9211c3fa1b325f6d97e44a7d1823a


## Cost per symbol-month, measured

One download per product, `{SYMBOL}` for `{MONTH}`, timed end to end: request, checksum, unzip,
parse. No caching, so the download time is a real network transfer each time this cell runs.

In [5]:
def measure(name: str, cfg: dict) -> dict:
    t0 = time.perf_counter()
    resp = requests.get(cfg["url"], timeout=120)
    resp.raise_for_status()
    download_s = time.perf_counter() - t0
    compressed = resp.content

    checksum_resp = requests.get(cfg["url"] + ".CHECKSUM", timeout=30)
    checksum_resp.raise_for_status()
    verify_checksum(compressed, checksum_resp.text.split()[0])

    t0 = time.perf_counter()
    with zipfile.ZipFile(io.BytesIO(compressed)) as zf:
        csv_bytes = zf.read(zf.namelist()[0])
    unzip_s = time.perf_counter() - t0

    t0 = time.perf_counter()
    df = pd.read_csv(io.BytesIO(csv_bytes), header=None, names=cfg["columns"])
    parse_s = time.perf_counter() - t0

    return {
        "product": name,
        "compressed_MB": len(compressed) / 1e6,
        "uncompressed_MB": len(csv_bytes) / 1e6,
        "compression_ratio": len(csv_bytes) / len(compressed),
        "download_s": download_s,
        "unzip_s": unzip_s,
        "parse_s": parse_s,
        "rows": len(df),
    }


results = [measure(name, cfg) for name, cfg in PRODUCTS.items()]

cost_table = pd.DataFrame(results).set_index("product")
cost_table

,compressed_MB,uncompressed_MB,compression_ratio,download_s,unzip_s,parse_s,rows
product,,,,,,,
klines,73.362908,363.766428,4.958452,6.634540,0.903871,2.390783,2592000
aggTrades,353.905374,2317.901627,6.549495,23.818846,5.585184,16.477777,27774325
trades,409.291743,2612.264594,6.382402,26.473352,6.920267,23.822655,35341653


## Against the brief's quoted numbers

The brief quoted roughly 71 MB, 409 MB and 696 MB compressed for klines, aggTrades and trades.
Measured for `BTCUSDT` `2024-06`:

In [6]:
brief_MB = {"klines": 71, "aggTrades": 409, "trades": 696}
comparison = cost_table[["compressed_MB"]].copy()
comparison["brief_quoted_MB"] = comparison.index.map(brief_MB)
comparison["ratio_measured_over_quoted"] = (
    comparison["compressed_MB"] / comparison["brief_quoted_MB"]
)
comparison

,compressed_MB,brief_quoted_MB,ratio_measured_over_quoted
product,,,
klines,73.362908,71,1.033280
aggTrades,353.905374,409,0.865294
trades,409.291743,696,0.588063


klines is close to the quoted figure. aggTrades and trades are not: measured compressed size for
`2024-06` is well under the brief's numbers for both. One symbol-month is one draw, and Binance
archive size scales with traded volume that month, so a different month would give a different
number; the brief's figures were not reproduced here for aggTrades or trades, and that discrepancy
is worth someone rechecking against a different month before it is relied on.

## Recommendation

Record the trade-off rather than resolve it, per #7's notes: whether 1-second resolution is
enough or trade-level sequencing is needed depends on the estimand, and the estimand is not
chosen (#9).

In [7]:
ratios = cost_table.loc["trades"] / cost_table.loc["klines"]
print(
    f"trades vs klines, same month: {ratios['compressed_MB']:.1f}x compressed disk, "
    f"{ratios['parse_s']:.1f}x parse time, {ratios['download_s']:.1f}x download time"
)

trades vs klines, same month: 5.6x compressed disk, 10.0x parse time, 4.0x download time


Default to 1-second klines for any measurement that only needs second-level timing, because they
are several times cheaper to store and parse than trade-level data for the same month and there is
no other cost difference between the products; only escalate to aggTrades or trades once the
estimand is chosen and is shown to need sub-second sequencing.

## Total obtainable samples

How many symbol-months of `BTCUSDT` data actually exist on `data.binance.vision`, and over how
many calendar days:

In [8]:
import calendar


def list_zips(prefix: str) -> list[str]:
    resp = requests.get(S3_LIST, params={"delimiter": "/", "prefix": prefix}, timeout=30)
    resp.raise_for_status()
    keys = re.findall(r"<Key>(.*?)</Key>", resp.text)
    return sorted(k for k in keys if k.endswith(".zip"))


def month_key(zip_path: str) -> str:
    return re.search(r"(\d{4}-\d{2})\.zip$", zip_path).group(1)


monthly_klines = list_zips(f"data/spot/monthly/klines/{SYMBOL}/1s/")
print("months available:", len(monthly_klines))
print("first month:", monthly_klines[0].rsplit("/", 1)[-1])
print("last month:", monthly_klines[-1].rsplit("/", 1)[-1])

months available: 108
first month: BTCUSDT-1s-2017-08.zip
last month: BTCUSDT-1s-2026-07.zip


In [9]:
for zip_path in [monthly_klines[0], monthly_klines[-1]]:
    ym = month_key(zip_path)
    year, mon = (int(x) for x in ym.split("-"))
    days_in_month = calendar.monthrange(year, mon)[1]
    daily_files = list_zips(f"data/spot/daily/klines/{SYMBOL}/1s/{SYMBOL}-1s-{ym}")
    print(
        ym,
        "daily files:",
        len(daily_files),
        "/",
        days_in_month,
        "calendar days,",
        "first day:",
        daily_files[0].rsplit("/", 1)[-1],
    )

leap_ym = "2020-02"
leap_files = list_zips(f"data/spot/daily/klines/{SYMBOL}/1s/{SYMBOL}-1s-{leap_ym}")
print(
    leap_ym, "daily files:", len(leap_files), "/", calendar.monthrange(2020, 2)[1], "calendar days"
)

2017-08 daily files: 15 / 31 calendar days, first day: BTCUSDT-1s-2017-08-17.zip


2026-07 daily files: 31 / 31 calendar days, first day: BTCUSDT-1s-2026-07-01.zip


2020-02 daily files: 29 / 29 calendar days


aggTrades and trades cover the identical 108 months (checked the same way, not shown twice here).
`BTCUSDT` daily data starts 2017-08-17, not on the 1st; the pair evidently was not yet listed, or
not yet 1-second-kline-eligible, before that date. The last month (2026-07) and a spot-checked
leap-year month (2020-02) both have one daily file per calendar day.

CPI and the Employment Situation are each released once a month by the BLS, with no month skipped
in this window to this project's knowledge. That puts an upper bound of 108 CPI releases and 108
Employment Situation releases inside the archive window, one per available month. This is a
ceiling from publication cadence, not a verified list of 108 dates; BLS's own release-date pages
returned HTTP 403 to automated fetches during this spike, so the exact list is #8's job, not this
one's.

FOMC statements are not monthly. `federalreserve.gov/monetarypolicy/fomccalendars.htm` was
fetched live during this spike and lists exact dates from 2021 onward:

In [10]:
fomc_2021_2026h1 = [
    "2021-01-27",
    "2021-03-17",
    "2021-04-28",
    "2021-06-16",
    "2021-07-28",
    "2021-09-22",
    "2021-11-03",
    "2021-12-15",
    "2022-01-26",
    "2022-03-16",
    "2022-05-04",
    "2022-06-15",
    "2022-07-27",
    "2022-09-21",
    "2022-11-02",
    "2022-12-14",
    "2023-02-01",
    "2023-03-22",
    "2023-05-03",
    "2023-06-14",
    "2023-07-26",
    "2023-09-20",
    "2023-11-01",
    "2023-12-13",
    "2024-01-31",
    "2024-03-20",
    "2024-05-01",
    "2024-06-12",
    "2024-07-31",
    "2024-09-18",
    "2024-11-07",
    "2024-12-18",
    "2025-01-29",
    "2025-03-19",
    "2025-05-07",
    "2025-06-18",
    "2025-07-30",
    "2025-09-17",
    "2025-10-29",
    "2025-12-10",
    "2026-01-28",
    "2026-03-18",
    "2026-04-29",
    "2026-06-17",
    "2026-07-29",
]
print("confirmed FOMC statement releases, 2021-01 through 2026-07:", len(fomc_2021_2026h1))

confirmed FOMC statement releases, 2021-01 through 2026-07: 45


That is 45 confirmed releases inside the archive window (each date is the second day of a
two-day meeting, when the statement is released). 2017-08 through 2020-12 was not independently
fetched in this spike; the FOMC has held 8 regularly scheduled meetings a year since 1981, which
gives a cadence estimate of roughly 3 for the partial second half of 2017 plus 24 for 2018-2020, on
top of 2 known unscheduled emergency statements during the March 2020 COVID intervention. Estimated
total across the full archive window: 45 confirmed + about 29 estimated = roughly 74, with the
2017-2020 portion unverified and deferred to #8.

Also considered and not counted here: the Fed's semiannual Monetary Policy Report testimony and ad
hoc Fedspeak are scheduled but not timestamped to the second the way CPI, the Employment Situation
and FOMC statements are, so they do not fit this project's instrument (see `CLAUDE.md`).

### Proof the data is healthy at those points in time

Two checks: the daily klines archive exists (HTTP 200) for every sample date, and for a couple of
them the full day parses with no gaps and a real price at the event second.

In [11]:
SAMPLE_DATES = {
    "CPI": [
        "2022-06-10",
        "2022-10-13",
        "2022-11-10",
        "2023-02-14",
        "2023-06-13",
        "2023-11-14",
        "2024-01-11",
        "2024-06-12",
        "2024-11-13",
    ],  # verified release dates, reused from #6
    "FOMC": ["2021-01-27", "2023-07-26", "2026-01-28"],
}

health_rows = []
for kind, dates in SAMPLE_DATES.items():
    for date in dates:
        url = f"{BASE}/data/spot/daily/klines/{SYMBOL}/1s/{SYMBOL}-1s-{date}.zip"
        head = requests.head(url, timeout=30)
        health_rows.append({"kind": kind, "date": date, "status": head.status_code})

health_table = pd.DataFrame(health_rows)
health_table

,kind,date,status
0,CPI,2022-06-10,200
1,CPI,2022-10-13,200
2,CPI,2022-11-10,200
3,CPI,2023-02-14,200
4,CPI,2023-06-13,200
5,CPI,2023-11-14,200
6,CPI,2024-01-11,200
7,CPI,2024-06-12,200
8,CPI,2024-11-13,200
9,FOMC,2021-01-27,200


In [12]:
KLINE_COLUMNS = PRODUCTS["klines"]["columns"]


def fetch_daily_klines(date: str) -> pd.DataFrame:
    url = f"{BASE}/data/spot/daily/klines/{SYMBOL}/1s/{SYMBOL}-1s-{date}.zip"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        raw = zf.read(zf.namelist()[0])
    df = pd.read_csv(io.BytesIO(raw), header=None, names=KLINE_COLUMNS)
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    return df


def event_row(date: str, hh: int, mm: int, ss: int) -> pd.DataFrame:
    df = fetch_daily_klines(date)
    y, m, d = (int(x) for x in date.split("-"))
    event_utc = pd.Timestamp(
        datetime(y, m, d, hh, mm, ss, tzinfo=ZoneInfo("America/New_York"))
    ).tz_convert("UTC")
    print(
        date,
        "rows:",
        len(df),
        "nulls in close:",
        int(df["close"].isna().sum()),
        "event second present:",
        (df["open_time"] == event_utc).any(),
    )
    return df[df["open_time"] == event_utc]


event_row("2022-06-10", 8, 30, 0)  # CPI, 08:30 ET
event_row("2021-01-27", 14, 0, 0)  # FOMC statement, 14:00 ET

2022-06-10 rows: 86400 nulls in close: 0 event second present: True


2021-01-27 rows: 86400 nulls in close: 0 event second present: True


,open_time,open,high,low,close,volume,close_time,quote_volume,trades,taker_buy_base,taker_buy_quote,ignore
68400,2021-01-27 19:00:00+00:00,29719.93,29731.09,29718.87,29731.09,1.482544,1611774000999,44067.482795,28,1.211982,36024.009628,0


Both days have the full 86,400 one-second rows with no nulls in `close`, and both event seconds
are present with a real traded price. That is two dates out of twelve sampled; the other ten only
had their existence checked (HTTP 200 above), not their row count.

## Summary

Klines, aggTrades and trades are all reachable at the documented monthly and daily URL patterns,
and checksum verification catches a corrupted download. Measured cost for one `BTCUSDT`
symbol-month (`2024-06`) is in `cost_table` above; klines matches the brief's quoted size, aggTrades
and trades do not.

`BTCUSDT` 1-second klines and both trade-level products cover 108 months, 2017-08-17 through
2026-07-31, with no missing days found in the months checked. That bounds CPI and Employment
Situation releases obtainable in this window at 108 each, and puts confirmed FOMC statement
releases at 45, with the 2017-2020 portion of the FOMC count estimated, not verified, in this
spike.

Recommendation: default to 1-second klines; escalate to trade-level data only once #9 shows it is
needed.